# Lab 06 — Forecasting：時間序列預測、預警與風險評估

異常偵測（Labs 02-04）回答「現在是否已經不正常」。本 lab 往前一步，用 **時間序列預測** 回答「如果趨勢持續，未來可能發生什麼，何時可能跨越風險門檻」，把監控從事後偵測提升為 **事前預警與風險評估**。

本 lab 對應理論單元 07，實作四件事：

1. 用 **Prophet** 建立可解釋的預測模型（趨勢 + 季節性 + 事件 + 誤差）。
2. 建立 **prediction interval** 並轉換為 **early warning**（threshold crossing、time-to-threshold、exceedance probability）。
3. 把 forecast 結果與 **SPC control limit** 整合，形成雙層告警（predictive + reactive）。
4. 輸出可用於 **Grafana 告警規則** 的風險訊號，透過 drop zone exporter 餵進 dashboard。

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab06_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab06_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab06_pipeline_position.svg")

## Lab 06 概覽

### 學習目標

1. 區分 **異常偵測、時間序列預測、預警、風險評估** 四種問題與輸出。
2. 用 **Prophet** 建立趨勢與多重季節性的可解釋預測 baseline。
3. 用 **holdout** 評估預測品質（MAE / RMSE / WAPE / MASE / 區間涵蓋率），並理解 rolling-origin / expanding / sliding 等更嚴謹的切分。
4. 把點預測與預測區間轉換為 **threshold crossing、time-to-threshold、exceedance probability** 與預警分級。
5. 整合 **Forecast + SPC**：對預測殘差做管制，形成 predictive 與 reactive 雙層告警。
6. 輸出 forecast risk signal，透過 **Prometheus drop zone** 在 Grafana 呈現與告警。

### 前置條件

本 lab **自給自足**：直接讀取 `data/synthetic/synthetic_rrd_metrics.csv` 並在 notebook 內計算特徵，輸出寫到 `outputs/workshop/`，不需要先跑其他 lab。額外套件 `prophet`、`scipy`（見 `environments/environment.week6.yml`，additive）。Grafana 整合為選用，需先完成 `labs/getting-started/05-prometheus-dropzone.md` 的一次性設定。

## 設計主線：預測要服務反應時間

Forecasting 的價值不在於把未來畫得漂亮，而在於 **在 SLA 受影響前給工程師足夠的反應時間（lead time）**。

$$Lead\ Time = t_{\text{predicted event}} - t_{\text{alert}}$$

設計預測型預警時請問三個問題：

1. **horizon 是否可行**：預測 30 分鐘後、2 小時後，對值班與自動化流程的意義不同。horizon 必須對齊「處置所需時間」，否則就算預測準也來不及。
2. **interval 是否對應風險**：區間太窄會誤報，太寬會漏報。預警應該看 **點預測 + 區間相對門檻的位置**，不是只看一條線。
3. **季節性是否穩定**：上週同時段是否真的能代表今天？節假日、搬遷、業務成長會破壞這個假設，這時要靠殘差監控接手。

設計原則：forecasting 是 **營運提前量** 的設計，不只是時間序列模型選型。

In [ ]:
from pathlib import Path
import warnings, logging
warnings.filterwarnings("ignore")
for name in ("cmdstanpy", "prophet", "stan"):     # Prophet 透過 cmdstanpy 擬合，關掉冗長 log
    logging.getLogger(name).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "workshop"     # workshop track output dir
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

try:
    from prophet import Prophet
    HAS_PROPHET = True
except Exception:
    HAS_PROPHET = False
    print("Prophet 未安裝，將自動退回 seasonal-naive baseline。安裝：conda install -c conda-forge prophet")

print(f"Project root : {PROJECT_ROOT}")
print(f"Prophet ready: {HAS_PROPHET}")

In [ ]:
def build_features(df):
    """Derive the semantic features used in this lab from the raw RRD counters (cf. Lab 01)."""
    df = df.sort_values(["port_id", "timestamp"]).reset_index(drop=True)
    sdiv = lambda a, b: np.where(b > 0, a / b, 0.0)
    df["traffic_total"]   = df["INOCTETS"] + df["OUTOCTETS"]
    df["ucast_total"]     = df["INUCASTPKTS"] + df["OUTUCASTPKTS"]
    df["broadcast_total"] = df["INBROADCASTPKTS"] + df["OUTBROADCASTPKTS"]
    df["multicast_total"] = df["INMULTICASTPKTS"] + df["OUTMULTICASTPKTS"]
    df["errors_total"]    = df["INERRORS"] + df["OUTERRORS"]
    df["discards_total"]  = df["INDISCARDS"] + df["OUTDISCARDS"]
    df["error_rate"]      = sdiv(df["errors_total"], df["ucast_total"])
    df["discard_rate"]    = sdiv(df["discards_total"], df["ucast_total"])
    df["unknown_total"]   = df["INUNKNOWNPROTOS"]
    df["avg_packet_size"] = sdiv(df["traffic_total"], df["ucast_total"])
    return df

raw = pd.read_csv(DATA_SYNTHETIC / "synthetic_rrd_metrics.csv", parse_dates=["timestamp"])
features = build_features(raw)
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

TARGET = "traffic_total"        # 預測目標：雙向總流量（bytes / 5 min）
PRIMARY_PORT = "port-id7431"    # backup-storage：含 large_file_backup 事件，適合示範容量預警
STEPS_PER_HOUR = 12
STEPS_PER_DAY = 288

print(f"rows={len(features):,}  ports={features.port_id.nunique()}  "
      f"span={features.timestamp.min()} -> {features.timestamp.max()}")
display(event_catalog[["event_id", "event_type", "port_id", "start_time", "end_time", "description"]])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：從偵測走向預警

同一份監控資料可以回答四個層次的問題，輸出與決策時間點都不同：

| 概念 | 核心問題 | 主要輸出 | 決策時間 |
|---|---|---|---|
| 異常偵測 | 現在是否偏離正常？ | 異常標記、anomaly score | 異常發生當下或之後 |
| 時間序列預測 | 未來數值可能是多少？ | 點預測 $\hat{y}_{t+h}$、預測區間 | 異常發生之前 |
| 預警 | 未來是否可能跨越風險門檻？ | 告警等級、預計跨越時間 | 保留反應時間 |
| 風險評估 | 發生機率與影響有多大？ | 風險分數、處置優先序 | 支援資源配置 |

**預警 = 預測 + 不確定性 + 風險門檻 + 可行動的提前量。** 預測值本身不等於告警：若預測磁碟 20 分鐘後耗盡、而擴容需要 30 分鐘，就應立即處理；若只是 CPU 略升但無服務風險，就不一定要叫醒值班。

### Point Forecast vs Prediction Interval

- **Point forecast** $\hat{y}_{t+h}$：單一估計，便於和門檻比較，但容易造成「預測很確定」的錯覺。
- **Prediction interval** $[L_{t+h}, U_{t+h}]$：未來 *單一觀測值* 可能落入的範圍（不是 confidence interval，後者只描述平均趨勢的不確定性）。
- horizon 越長，誤差累積、外生事件更難掌握，區間通常越寬。若長期區間異常狹窄，多半是模型 **低估了不確定性**。

## 📖 概念：Prophet 的結構

Prophet 把時間序列分解成可解釋的幾個組成：

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

- $g(t)$：**趨勢**（分段線性或 logistic growth），用 changepoints 容許不同時段不同斜率。
- $s(t)$：**季節性**（日 / 週 / 年，用 Fourier series 表示）。網路流量常見日週期與週週期。
- $h(t)$：**節假日 / 已知事件**（部署、促銷、結帳、維護、大型批次）。
- $\epsilon_t$：無法解釋的誤差。

Prophet 的定位不是「最強模型」，而是 **適合具有明顯趨勢、多重季節性、變點與已知事件的可解釋 baseline**：建立快、結果可拆解。限制是對極短期高頻強自相關序列未必最佳、結構突變後需重訓、區間建立在歷史誤差假設上。這也是為什麼我們不會只信 Prophet：稍後用 **殘差 SPC** 補上「模型預期被打破」的偵測。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1 — 用 Prophet 建立預測模型與 prediction interval

先固定看 `PRIMARY_PORT`（backup-storage），對 `traffic_total` 擬合 Prophet，開啟日 / 週季節性，用 `interval_width=0.95` 產生 95% prediction interval。我們也把事件目錄裡這個 port 的已知事件做成 binary regressor 餵給 Prophet 當 **h(t)**，讓模型把事件視為可解釋變數，而不是讓它污染趨勢與殘差。擬合需要約 10-20 秒。

In [ ]:
def to_prophet_frame(df_port, target=TARGET):
    """Build a Prophet frame (ds, y) plus a binary `event` regressor = h(t).

    Known events for this port (from the catalog) become an explanatory variable,
    so Prophet models them as h(t) instead of letting them pollute the trend/residual."""
    f = (df_port[["timestamp", target]].rename(columns={"timestamp": "ds", target: "y"})
         .dropna().sort_values("ds").reset_index(drop=True))
    pid = df_port["port_id"].iloc[0]
    ev = event_catalog[(event_catalog.port_id == pid) | (event_catalog.port_id == "MULTI")]
    mask = pd.Series(False, index=f.index)
    for _, e in ev.iterrows():
        mask |= (f["ds"] >= e["start_time"]) & (f["ds"] <= e["end_time"])
    f["event"] = mask.astype(int)
    return f

def fit_prophet(train_df, interval_width=0.95):
    """Fit Prophet with daily+weekly seasonality + a known-event regressor h(t) (MAP estimate)."""
    m = Prophet(growth="linear", daily_seasonality=True, weekly_seasonality=True,
                yearly_seasonality=False, seasonality_mode="additive",
                interval_width=interval_width, changepoint_prior_scale=0.05,
                uncertainty_samples=300)
    if "event" in train_df.columns:
        m.add_regressor("event")     # h(t): known events as an explanatory variable
    m.fit(train_df)
    return m

def seasonal_naive_forecast(series_df, width=0.95):
    """Fallback when Prophet is unavailable: same-slot history + empirical interval."""
    g = series_df.sort_values("ds").reset_index(drop=True).copy()
    g["slot"] = (g["ds"].dt.dayofweek.astype(str) + "-" + g["ds"].dt.hour.astype(str)
                 + "-" + (g["ds"].dt.minute // 5).astype(str))
    g["yhat"] = g.groupby("slot")["y"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    g["yhat"] = g["yhat"].fillna(g["y"].expanding().mean())
    resid = (g["y"] - g["yhat"]).abs()
    band = resid.rolling(STEPS_PER_DAY, min_periods=STEPS_PER_HOUR).quantile(width).bfill().fillna(resid.quantile(width))
    g["yhat_lower"] = g["yhat"] - band
    g["yhat_upper"] = g["yhat"] + band
    return g[["ds", "yhat", "yhat_lower", "yhat_upper"]]

port_df = features[features.port_id == PRIMARY_PORT].copy()
series = to_prophet_frame(port_df)

if HAS_PROPHET:
    model = fit_prophet(series)
    forecast_in = model.predict(series[["ds", "event"]])
    print(f"Fitted Prophet on {len(series):,} points for {PRIMARY_PORT} (with event regressor h(t))")
else:
    model = None
    forecast_in = seasonal_naive_forecast(series)

fc = series.merge(forecast_in[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds", how="left")
fc["yhat_lower"] = fc["yhat_lower"].clip(lower=0)
display(fc[["ds", "y", "yhat", "yhat_lower", "yhat_upper"]].head())

## Step 2 — 視覺化：Actual vs Forecast + prediction interval

In [ ]:
def shade_events(ax, port_id, catalog):
    for _, ev in catalog.iterrows():
        if ev["port_id"] in (port_id, "MULTI"):
            ax.axvspan(ev["start_time"], ev["end_time"], alpha=0.12, color="crimson", zorder=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fc["ds"], fc["y"], color="lightgray", lw=0.8, label="actual")
ax.plot(fc["ds"], fc["yhat"], color="steelblue", lw=1.2, label="Prophet forecast" if HAS_PROPHET else "seasonal-naive")
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.20, color="steelblue", label="95% prediction interval")
shade_events(ax, PRIMARY_PORT, event_catalog)
ax.set_title(f"{PRIMARY_PORT} — {TARGET}: actual vs forecast (red shading = known event)")
ax.set_ylabel("Bytes / 5 min"); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); show_fig(fig)

if HAS_PROPHET:
    comp = model.plot_components(forecast_in)
    comp.set_size_inches(11, 6)
    show_fig(comp)

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：時間序列不能隨機切分

預測模型必須遵守時間順序：用未來資料預測過去會造成 **data leakage**。合理評估是 **rolling-origin / holdout**：用前段訓練、後段測試。

常見指標（$y_t$ 實際、$\hat{y}_t$ 預測）：

- **MAE** $=\frac{1}{n}\sum|y_t-\hat{y}_t|$：與原始單位相同、易解釋。
- **RMSE** $=\sqrt{\frac{1}{n}\sum(y_t-\hat{y}_t)^2}$：對大誤差懲罰較重。
- **WAPE** $=\frac{\sum|y_t-\hat{y}_t|}{\sum|y_t|}$：適合整體量級比較，避免 MAPE 在低值處爆炸。
- **MASE**：模型誤差除以 naive forecast 誤差。**小於 1 代表優於 naive baseline**，是「值不值得用複雜模型」的關鍵檢查。
- **Coverage**：宣稱 95% 的區間，實際涵蓋率應接近 95%。過低代表過度自信，過高且過寬則失去決策價值。

> 模型 MAE 小不代表預警一定好。預警還要看事件召回、提前量、誤警與漏警（見結尾的人類驗證點）。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 — Holdout 評估：預測品質

用最後 3 天當測試集（holdout，**單一切分**；rolling-origin 會把切分點逐步向前移動再平均，更嚴謹但更花時間），其餘訓練。比較 Prophet 與 **seasonal-naive baseline**（同時段一週前的值），計算 MAE / RMSE / WAPE / MASE 與區間涵蓋率。

In [ ]:
HOLDOUT_DAYS = 3
cut = series["ds"].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train, test = series[series.ds <= cut], series[series.ds > cut]

if HAS_PROPHET:
    m_eval = fit_prophet(train)
    pred = m_eval.predict(test[["ds", "event"]])[["ds", "yhat", "yhat_lower", "yhat_upper"]]
else:
    full = seasonal_naive_forecast(series)
    pred = full[full.ds.isin(test.ds)]
ev = test.merge(pred, on="ds", how="inner")

sn = series.copy()
sn["yhat_sn"] = sn["y"].shift(7 * STEPS_PER_DAY)         # 同時段一週前
ev = ev.merge(sn[["ds", "yhat_sn"]], on="ds", how="left")
ev["yhat_sn"] = ev["yhat_sn"].ffill().bfill()

def mae(y, yh):  return float(np.mean(np.abs(y - yh)))
def rmse(y, yh): return float(np.sqrt(np.mean((y - yh) ** 2)))
def wape(y, yh): return float(np.sum(np.abs(y - yh)) / max(np.sum(np.abs(y)), 1e-9))

model_mae, naive_mae = mae(ev.y, ev.yhat), mae(ev.y, ev.yhat_sn)
naive1_mae = float(np.mean(np.abs(np.diff(train["y"].values))))   # in-sample one-step naive = MASE denominator
coverage = float(((ev.y >= ev.yhat_lower) & (ev.y <= ev.yhat_upper)).mean())
metrics = pd.DataFrame({
    "MAE":  [model_mae, naive_mae],
    "RMSE": [rmse(ev.y, ev.yhat), rmse(ev.y, ev.yhat_sn)],
    "WAPE": [wape(ev.y, ev.yhat), wape(ev.y, ev.yhat_sn)],
    "MASE": [model_mae / max(naive1_mae, 1e-9), naive_mae / max(naive1_mae, 1e-9)],   # /in-sample naive MAE; <1 beats naive
}, index=["Prophet" if HAS_PROPHET else "seasonal-roll", "seasonal-naive (1wk)"])
print(f"95% prediction-interval coverage on holdout: {coverage:.1%}  (target ~ 95%)")
display(metrics.round(3))

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(ev.ds, ev.y, color="black", lw=1, label="actual")
ax.plot(ev.ds, ev.yhat, color="steelblue", lw=1.2, label="forecast")
ax.fill_between(ev.ds, ev.yhat_lower, ev.yhat_upper, alpha=0.2, color="steelblue", label="95% PI")
ax.set_title(f"{PRIMARY_PORT} — holdout ({HOLDOUT_DAYS}d): actual vs forecast")
ax.legend(loc="upper left", fontsize=9); plt.tight_layout(); show_fig(fig)

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：從預測值轉換為風險

有了 $\hat{y}_{t+h}$ 與區間 $[L,U]$，就能把預測轉成風險語言：

1. **Threshold crossing**：預測值何時跨越門檻 $T$（容量 / SLO / 規格 / SPC limit）：$\hat{y}_{t+h}\ge T$。
2. **Time-to-threshold**：最早跨越的時間 $\tau=\min\{h:\hat{y}_{t+h}\ge T\}$，回答「還有多久出事」。
3. **Exceedance probability**：把預測視為 $Y_{t+h}\sim\mathcal{N}(\hat{y},\sigma)$（$\sigma$ 由區間反推），算 $P(Y_{t+h}>T)$。

### 區間式風險判讀

| 狀態 | 判讀 | 意義 |
|---|---|---|
| 低風險 Info | 上界 $U<T$ | 目前不確定性下不太可能超線 |
| 觀察 Watch | $U\ge T$ 但 $\hat{y}<T$ | 有風險可能，需持續監控 |
| 高風險 Warning | $\hat{y}\ge T$ | 典型未來路徑已超線 |
| 極高風險 Critical | 下界 $L\ge T$ | 多數合理未來情境都將超線 |

### 風險分數

$$RiskScore = f(P_{\text{exceed}}, Severity, Urgency, Confidence, Actionability)$$

預警分級（Info / Watch / Warning / Critical）應反映 **處置需求**，不只是數值大小。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 — Threshold crossing、time-to-threshold、exceedance probability

定義 capacity 門檻（高分位數 × 安全係數），對 `PRIMARY_PORT` 做 **向前預測**（horizon = 30 分鐘），計算 time-to-threshold、exceedance probability 與區間風險狀態。資料近似平穩時，point forecast 可能不會在 horizon 內跨越門檻（time-to-threshold = None）：這正是要看 **區間風險分級** 與 **exceedance probability**，而不是只等點預測跨線。

In [ ]:
FORECAST_HORIZON_STEPS = STEPS_PER_HOUR // 2     # 6 steps x 5 min = 30 min (matches forecast_30m + predict_linear 1800s)
INTERVAL_Z = stats.norm.ppf(0.975)        # 95% 區間對應的 z (~1.96)

def capacity_for(s, q=0.97, factor=1.05):
    # teaching threshold: just above the 97th percentile so the risk ladder actually lights up.
    # in production replace with the engineered capacity / SLO limit.
    return float(s["y"].quantile(q) * factor)

def sigma_from_interval(upper, lower, z=INTERVAL_Z):
    return np.maximum((upper - lower) / (2 * z), 1e-9)

def exceedance_prob(yhat, sigma, T):
    return stats.norm.sf(T, loc=yhat, scale=sigma)

capacity = capacity_for(series)
if HAS_PROPHET:
    future = model.make_future_dataframe(periods=FORECAST_HORIZON_STEPS, freq="5min", include_history=False)
    future["event"] = 0          # no known upcoming event -> h(t) contributes 0 going forward
    fwd = model.predict(future)[["ds", "yhat", "yhat_lower", "yhat_upper"]]
else:
    fwd = seasonal_naive_forecast(series).tail(FORECAST_HORIZON_STEPS).copy()
fwd["yhat_lower"] = fwd["yhat_lower"].clip(lower=0)
fwd["sigma"] = sigma_from_interval(fwd["yhat_upper"], fwd["yhat_lower"])
fwd["p_exceed"] = exceedance_prob(fwd["yhat"].values, fwd["sigma"].values, capacity)
fwd["horizon_min"] = np.arange(1, len(fwd) + 1) * 5

def risk_state(r, T=capacity):
    if r["yhat_lower"] >= T: return "Critical (極高風險)"
    if r["yhat"]       >= T: return "Warning (高風險)"
    if r["yhat_upper"] >= T: return "Watch (觀察)"
    return "Info (低風險)"
fwd["risk_state"] = fwd.apply(risk_state, axis=1)

crossed = fwd[fwd["yhat"] >= capacity]
ttt = int(crossed["horizon_min"].iloc[0]) if not crossed.empty else None
print(f"capacity threshold T = {capacity:,.0f} bytes/5min")
print(f"time-to-threshold (point forecast crosses T): {ttt} min" if ttt else "point forecast does not cross T within horizon")
print(f"max exceedance probability over horizon: {fwd['p_exceed'].max():.1%}")
display(fwd[["ds", "yhat", "yhat_upper", "horizon_min", "p_exceed", "risk_state"]])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：Forecast + SPC 整合

SPC（Lab 03）關心「現在是否偏離正常統計行為」；forecasting 關心「未來可能走向何處」。整合後兩者互補：

| 方法 | 主要比較對象 | 核心問題 |
|---|---|---|
| SPC | 實際值 vs 管制界限 | 系統現在是否失控？ |
| Forecasting | 未來預測 vs 風險門檻 | 系統未來是否可能失控？ |
| **Forecast residual SPC** | 實際值 vs 預測值 | **模型的預期是否被打破？** |

### 以殘差進行 SPC

定義殘差 $e_t = y_t - \hat{y}_t$。模型若已解釋趨勢與季節性，殘差應接近平穩。對殘差套用 Shewhart（$\bar{e}\pm3\sigma_e$）或 EWMA，可偵測突發偏離、模型未捕捉的新型態、結構改變與系統性偏誤，補上 Prophet 看不到的「正常季節性以外的突發」。

### 雙層告警

- **Reactive alert**：實際值已超限或殘差出管制（現在就出事了）。
- **Predictive alert**：預測值將在未來跨越門檻（還沒出事，附帶預計時間與不確定性）。

兩者 **不應是同一個嚴重度**。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 — Forecast 殘差 SPC 與雙層告警

對 in-sample 殘差建立 Shewhart 與 EWMA 管制界限，標記 **reactive**（殘差出管制）與 **predictive**（向前預測接近門檻）兩種告警。

注意兩件事：(1) `predictive_alert` 用「區間上界 > capacity × 0.8」作為 **提前邊際**，與 Step 4 的區間風險分級（上界/點/下界 對 T）是兩套互補判讀，不要混為一談；(2) 殘差 SPC 界限與 `p_exceed` 都用 **in-sample 殘差** 估計，會略為樂觀（真實 out-of-sample 誤差通常更大），可改用 Step 3 holdout 殘差校準。

In [ ]:
fc = fc.sort_values("ds").reset_index(drop=True)
fc["residual"] = fc["y"] - fc["yhat"]

r_center = fc["residual"].median()
r_sigma = 1.4826 * (fc["residual"] - r_center).abs().median()    # MAD-based robust sigma
fc["resid_z"] = (fc["residual"] - r_center) / max(r_sigma, 1e-9)
fc["spc_shewhart"] = fc["resid_z"].abs() > 3

lam = 0.2
fc["ewma"] = fc["residual"].ewm(alpha=lam, adjust=False).mean()
ewma_sigma = r_sigma * np.sqrt(lam / (2 - lam))
fc["spc_ewma"] = (fc["ewma"] - r_center).abs() > 3 * ewma_sigma

fc["reactive_alert"] = fc["spc_shewhart"] | fc["spc_ewma"]
fc["capacity"] = capacity
fc["predictive_alert"] = fc["yhat_upper"] > capacity * 0.80

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(fc.ds, fc.y, color="lightgray", lw=0.7, label="actual")
axes[0].plot(fc.ds, fc.yhat, color="steelblue", lw=1.0, label="forecast")
axes[0].axhline(capacity, color="tab:red", ls="--", lw=1, label="capacity T")
react = fc[fc.reactive_alert]
axes[0].scatter(react.ds, react.y, s=14, color="tab:red", zorder=5, label="reactive (residual SPC)")
shade_events(axes[0], PRIMARY_PORT, event_catalog)
axes[0].set_title(f"{PRIMARY_PORT} — forecast + capacity + reactive alerts"); axes[0].legend(loc="upper left", fontsize=8)

axes[1].plot(fc.ds, fc.resid_z, color="slategray", lw=0.7, label="residual z")
axes[1].axhline(3, color="tab:red", ls=":"); axes[1].axhline(-3, color="tab:red", ls=":")
axes[1].set_title("Forecast residual z-score (Shewhart ±3σ)"); axes[1].set_ylabel("z")
shade_events(axes[1], PRIMARY_PORT, event_catalog)
plt.tight_layout(); show_fig(fig)

print(f"reactive alerts (residual out-of-control): {int(fc.reactive_alert.sum())}")
print(f"predictive-warning points (interval nears capacity): {int(fc.predictive_alert.sum())}")

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：把風險訊號送進 Grafana（drop zone，additive）

跟 Lab 08 一樣，notebook 邏輯有一條對應的 **生產軌道**。這門課用 **Prometheus drop zone** 把 notebook 結果餵進 Grafana，完全 **不需要改 Prometheus / Grafana 設定**：

```text
notebook 寫出 forecast_results.csv
  -> 複製到 outputs/prometheus-dropzone/current_results.csv
  -> infra/python_results_exporter.py 讀檔，曝露 http://localhost:8010/metrics
  -> Prometheus 抓 job="python-results-exporter"
  -> Grafana 查 aiops_python_result
```

**全部都是加上去，不破壞既有設定**：

- 原本的 `infra/grafana/dashboards/network_metrics.json`（uid `aiops-network-rrd`）**保持不動**；本 lab 另外提供 **一個新的 dashboard** `infra/grafana/dashboards/forecast_rca_risk.json`（uid `aiops-forecast-rca`）專看 forecast / RCA 風險。
- 原本的 `infra/prometheus/alerts.yml` **保持不動**；若要 server 端的預測告警，新規則放在新檔 `infra/prometheus/forecast_alerts.yml`（用 `predict_linear` 做資料庫端容量預測 + 殘差 z 反應式告警），只在 `prometheus.yml` 的 `rule_files:` 多加一行即可。

延伸閱讀：Grafana Cloud ML forecasting <https://grafana.com/docs/grafana-cloud/machine-learning/dynamic-alerting/forecasting/> 與 dynamic alerting / RBAC <https://grafana.com/docs/grafana-cloud/machine-learning/dynamic-alerting/rbac/>。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 6 — 輸出 forecast risk signal（drop-zone 相容）

對全部 port 擬合預測模型，產生統一的 forecast risk signal 表，存成 `forecast_results.csv`。欄位 **保留 drop zone exporter 認得的欄位**（`y_hat`、`y_hat_lower`、`y_hat_upper`、`forecast_30m`、`early_warning_30m`）並 **加上** 風險欄位（`forecast_risk_score`、`resid_z`、`reactive_alert`、`p_exceed`、`warning_level`）。另存 `forecast_risk_signals.csv`（每 port 的 forward-horizon 風險彙整）。

In [ ]:
def build_port_forecast(pdf, target=TARGET):
    s = to_prophet_frame(pdf, target)
    if len(s) < 2 * STEPS_PER_DAY:
        return None
    cap = capacity_for(s)
    try:
        if not HAS_PROPHET:
            raise RuntimeError("no prophet")
        m = fit_prophet(s)
        ins = m.predict(s[["ds", "event"]])[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        method = "prophet"
    except Exception:
        ins = seasonal_naive_forecast(s)
        method = "seasonal_naive"
    g = s.merge(ins, on="ds", how="left")
    g["yhat_lower"] = g["yhat_lower"].clip(lower=0)
    g["residual"] = g["y"] - g["yhat"]
    c = g["residual"].median(); sg = 1.4826 * (g["residual"] - c).abs().median()
    g["resid_z"] = (g["residual"] - c) / max(sg, 1e-9)
    g["reactive_alert"] = (g["resid_z"].abs() > 3).astype(int)
    g["capacity"] = cap
    g["sigma"] = sigma_from_interval(g["yhat_upper"], g["yhat_lower"])
    g["p_exceed"] = exceedance_prob(g["yhat"].values, g["sigma"].values, cap)
    g["forecast_30m"] = g["yhat"].shift(-FORECAST_HORIZON_STEPS)        # 模型對 30 分鐘後的點預測
    g["early_warning_30m"] = (g["forecast_30m"] > cap * 0.80).astype(int)
    g["forecast_positive_anomaly"] = (g["y"] > g["yhat_upper"]).astype(int)
    g["forecast_negative_anomaly"] = (g["y"] < g["yhat_lower"]).astype(int)
    def lvl(r):
        if r.yhat_lower >= cap: return "Critical"
        if r.yhat       >= cap: return "Warning"
        if r.yhat_upper >= cap: return "Watch"
        return "Info"
    g["warning_level"] = g.apply(lvl, axis=1)
    react_component = np.clip(g["resid_z"].abs() / 6.0, 0, 1)
    g["forecast_risk_score"] = (100 * (0.6 * g["p_exceed"] + 0.4 * react_component)).round(1)
    # carry label columns expected by the drop-zone exporter
    meta = (pdf[["timestamp", "device_id", "port_id", "port_role", "event_label"]]
            .rename(columns={"timestamp": "ds"}))
    g = g.merge(meta, on="ds", how="left")
    g["ml_method"] = method
    # forward-horizon summary
    if HAS_PROPHET and method == "prophet":
        future_ = m.make_future_dataframe(periods=FORECAST_HORIZON_STEPS, freq="5min", include_history=False)
        future_["event"] = 0
        fwd_ = m.predict(future_)[["ds", "yhat", "yhat_lower", "yhat_upper"]]
    else:
        fwd_ = seasonal_naive_forecast(s).tail(FORECAST_HORIZON_STEPS)
    fwd_ = fwd_.copy(); fwd_["yhat_lower"] = fwd_["yhat_lower"].clip(lower=0)
    fwd_["sigma"] = sigma_from_interval(fwd_["yhat_upper"], fwd_["yhat_lower"])
    fwd_["p_exceed"] = exceedance_prob(fwd_["yhat"].values, fwd_["sigma"].values, cap)
    crossed_ = fwd_[fwd_["yhat"] >= cap]
    summary = {
        "port_id": pdf["port_id"].iloc[0], "device_id": pdf["device_id"].iloc[0],
        "port_role": pdf["port_role"].iloc[0], "ml_method": method, "capacity": round(cap, 1),
        "horizon_min": FORECAST_HORIZON_STEPS * 5,
        "time_to_threshold_min": int((crossed_["ds"].iloc[0] - s["ds"].max()).total_seconds() // 60) if not crossed_.empty else None,
        "max_p_exceed": round(float(fwd_["p_exceed"].max()), 3),
        "reactive_alerts": int(g["reactive_alert"].sum()),
        "max_forecast_risk_score": float(g["forecast_risk_score"].max()),
    }
    out_cols = ["ds", "device_id", "port_id", "port_role", "event_label", "ml_method", "y",
                "y_hat", "y_hat_lower", "y_hat_upper", "capacity", "forecast_30m", "early_warning_30m",
                "forecast_positive_anomaly", "forecast_negative_anomaly",
                "residual", "resid_z", "reactive_alert", "p_exceed", "warning_level", "forecast_risk_score"]
    g = g.rename(columns={"yhat": "y_hat", "yhat_lower": "y_hat_lower", "yhat_upper": "y_hat_upper"})
    for c in ("y_hat", "y_hat_lower", "y_hat_upper"):    # byte counts can't be negative
        g[c] = g[c].clip(lower=0)
    return g[out_cols].rename(columns={"ds": "timestamp", "y": target}), summary

all_fc, summaries = [], []
for pid, pdf in features.groupby("port_id", sort=False):
    res = build_port_forecast(pdf.copy())
    if res is None:
        continue
    gtab, summ = res
    all_fc.append(gtab); summaries.append(summ)
    print(f"  {pid} [{summ['ml_method']}]: max risk {summ['max_forecast_risk_score']:.1f}, "
          f"reactive {summ['reactive_alerts']}, max P(exceed) {summ['max_p_exceed']:.2f}")

forecast_results = pd.concat(all_fc, ignore_index=True)
risk_signals = pd.DataFrame(summaries).sort_values("max_forecast_risk_score", ascending=False)
forecast_results.to_csv(DATA_PROCESSED / "forecast_results.csv", index=False)
risk_signals.to_csv(DATA_PROCESSED / "forecast_risk_signals.csv", index=False)
print(f"\nsaved: forecast_results.csv ({len(forecast_results):,} rows), forecast_risk_signals.csv")
display(risk_signals)

### （選用）產生 server 端預測告警規則

下面這格把每 port 的 capacity 門檻寫成一份 **additive** 的 Prometheus 規則檔到 `outputs/`，內容與 committed 的 `infra/prometheus/forecast_alerts.yml` 等價（`predict_linear` 容量預測 + 殘差 z 反應式告警）。它 **不會動到** `alerts.yml`。

In [ ]:
import yaml
cap_map = {r.port_id: float(r.capacity) for r in risk_signals.itertuples()}
rules_doc = {"groups": [
    {"name": "network_forecast", "interval": "5s", "rules": [
        {"record": "network:traffic_predict_30m", "expr": "predict_linear(network:traffic_total[1h], 1800)"},
        {"record": "network:traffic_resid_z",
         "expr": "(network:traffic_total - network:traffic_mean_1h) / clamp_min(network:traffic_stddev_1h, 1)"},
    ]},
    {"name": "network_forecast_alerts", "interval": "5s", "rules":
        [{"alert": f"PredictiveCapacityWarning_{pid.replace('-', '_')}",
          "expr": f'network:traffic_predict_30m{{port_id="{pid}"}} > {cap*0.8:.0f}', "for": "10m",
          "labels": {"severity": "warning", "alert_kind": "predictive", "port_id": pid},
          "annotations": {"summary": f"Predicted traffic on {pid} approaches capacity within 30m"}}
         for pid, cap in cap_map.items()] +
        [{"alert": "ForecastResidualAnomaly", "expr": "abs(network:traffic_resid_z) > 3", "for": "1m",
          "labels": {"severity": "warning", "alert_kind": "reactive"},
          "annotations": {"summary": "Traffic deviates from forecast on {{ $labels.port_id }}"}}]},
]}
out_rules = DATA_PROCESSED / "forecast_alerts.generated.yml"
with open(out_rules, "w") as fh:
    yaml.safe_dump(rules_doc, fh, sort_keys=False, allow_unicode=True, width=100)
print(f"saved additive rules: {out_rules}")
print("wire (without editing alerts.yml): add  - \"forecast_alerts.yml\"  to prometheus.yml rule_files")

## Optional — 在 Grafana 看 forecast 風險（drop zone）

Notebook 已經顯示 forecast 與 prediction interval。若想在 Grafana 看，保持 `python infra/python_results_exporter.py` 執行，然後複製：

```bash
cp outputs/workshop/forecast_results.csv outputs/prometheus-dropzone/current_results.csv
```

建議 PromQL（這些欄位 exporter 預設就會曝露）：

```promql
aiops_python_result{column="y_hat"}
aiops_python_result{column="forecast_30m"}
aiops_python_result{column="early_warning_30m"}
```

要額外看風險欄位（`forecast_risk_score`、`resid_z`、`p_exceed`），啟動 exporter 時加上：

```bash
RESULTS_CSV_PATH=outputs/workshop/forecast_results.csv \
RESULT_COLUMNS=y_hat,forecast_30m,early_warning_30m,forecast_risk_score,resid_z,p_exceed \
python infra/python_results_exporter.py
```

匯入新的 dashboard `infra/grafana/dashboards/forecast_rca_risk.json`（uid `aiops-forecast-rca`）即可看到對應 panel。原本的 `aiops-network-rrd` dashboard 不受影響。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。

---

## ⚖️ 方法比較：Prophet vs predict_linear vs Rolling baseline vs SARIMA / LSTM

| | predict_linear (Prometheus) | Rolling historical avg | Prophet | SARIMA | LSTM |
|---|---|---|---|---|---|
| **趨勢** | 線性外插（短期） | 差 | 分段線性 + changepoints | MA 處理 | 好 |
| **季節性** | 無 | 同時段歷史（隱含日 / 週） | 日 / 週 / 年 Fourier | S 項 | 需大量資料 |
| **prediction interval** | 無 | 經驗分位數 | 內建 | 內建 | 需額外建模 |
| **可解釋性** | 高 | 高 | 高（可拆解組成） | 中 | 低 |
| **資料需求** | 數小時 | 1-4 週 | 數週 | 數月 | 數月到數年 |
| **落地位置** | 資料庫端即時 | notebook / 服務 | 模型服務 + drop zone | 模型服務 | MLOps 平台 |

### 何時升級

- 只關心 **容量耗盡（單調趨勢）**：`predict_linear` 已經夠用，且最省事。
- 有明顯 **日 / 週季節性** 與已知事件：用 **Prophet**（本 lab）。
- 長期穩定多季節 + 充足歷史：可評估 SARIMA。
- 模式高度非線性、資料量大、有 GPU 與 MLOps：才考慮 LSTM 或時序大模型。

**本課程的選擇**：Prophet 兼顧可解釋與季節性；搭配殘差 SPC 補上突發偵測，再用 `predict_linear` 做資料庫端容量預警。複雜模型若無法穩定勝過 naive baseline（MASE < 1），就不一定值得部署。

## ⚠ 人類驗證點 #6 — 預警的取捨：interval 寬度、提前量與不確定性

預測型預警的品質不只看「預測準不準」，而是看 **誤警、漏警、提前量與不確定性之間的決策權衡**。

### 如何判斷

1. **區間涵蓋率**：Step 3 的 coverage 接近 95% 嗎？過低代表過度自信（會漏警），過高且過寬代表預警長期模糊。
2. **MASE**：模型有勝過 naive baseline（< 1）嗎？沒有的話，複雜模型不一定值得維護。
3. **提前量 vs 處置時間**：`time_to_threshold_min` 是否大於實際擴容 / 切換所需時間？預測一小時後超線、但處置要四小時，預警就沒有價值。

### 讓你重新考慮的信號

- 正常時段每天觸發很多 predictive alert → 門檻或 horizon 太敏感。
- 已知的 `large_file_backup` / `queue_congestion` 事件完全沒被殘差 SPC 標到 → 殘差中心 / 尺度被事件污染，或季節性把事件學成了正常。
- 每次 forecast 更新跨線時間就跳動 → 需要 hysteresis、持續條件（`for:`）與 deduplication（接 Lab 05 / Lab 08）。

### 模型本身也要被監控

誤差是否增加、bias 是否漂移、區間涵蓋率是否下降、輸入是否漂移、預警頻率是否突變。預測模型部署後不是一勞永逸。

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習

在 Step 4 / Step 5 的 code cell 調整這些參數，重新執行並觀察：

```python
FORECAST_HORIZON_STEPS = 6    # 30 分 -> 改成 12（1 小時）或 24（2 小時），看 time-to-threshold 與區間怎麼變
INTERVAL_Z = 1.96             # 95% -> 改成 2.58（99%），區間變寬、coverage 與誤警怎麼變
changepoint_prior_scale=0.05  # fit_prophet 內：調大趨勢更敏感（追雜訊），調小更平滑（追不上變點）
predictive_alert: yhat_upper > capacity * 0.80   # 改 0.70 / 0.90，predictive alert 數量怎麼變
```

> 「如果公司把業務從一個資料中心遷到另一個，Prophet 的趨勢與季節性會在哪裡失效？殘差 SPC 會先抓到嗎？」

> 「`predict_linear` 與 Prophet 各自適合哪種預警情境？為什麼容量耗盡常常用前者就夠？」

> 「預測誤差最大的 port 是哪一個？這代表模型問題，還是那個 port 本來就最難預測？」

## 整合回顧：Lab 06 在 Pipeline 裡的位置

```
What you did in Lab 06                 Production equivalent
──────────────────────────            ─────────────────────────────────
fit_prophet()                  →       forecasting service / Grafana ML
prediction interval            →       y_hat_upper / y_hat_lower (drop zone -> aiops_python_result)
time-to-threshold / p_exceed   →       PredictiveCapacityWarning alert rule
residual SPC (Shewhart/EWMA)   →       ForecastResidualAnomaly alert rule
forecast_results.csv           →       drop zone -> Prometheus -> new Grafana dashboard
forecast_alerts.yml            →       infra/prometheus/ rule file (additive)

⚠ Human verification point #6  →       on-call tunes horizon / interval / lead time
```

**Lab 06 輸出**（寫到 `outputs/workshop/`，不污染 repo）：

- `forecast_results.csv` — 每 port 每時間點的 forecast、區間、殘差 z、risk score（drop-zone 相容）。
- `forecast_risk_signals.csv` — 每 port 的 forward-horizon 風險彙整（time-to-threshold、P(exceed)）。
- `forecast_alerts.generated.yml` — 與 committed 的 `infra/prometheus/forecast_alerts.yml` 等價的規則檔。

**新增的 Grafana / Prometheus 檔（additive，原檔不動）**：`infra/grafana/dashboards/forecast_rca_risk.json`、`infra/prometheus/forecast_alerts.yml`。

**下一步 → Lab 07 — Root Cause Analysis**：預測告訴你「未來可能何時出事」，RCA 接著回答「為什麼會出事」。當風險升高時，RCA 分析哪些上游變數先變化、哪些依賴元件可能造成風險。